In [36]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [51]:
df = pd.read_csv('https://raw.githubusercontent.com/renatomaaliw3/public_files/refs/heads/master/Data%20Sets/loan_approval.csv')
df.head(30)

,Age_Group,Income_Level,Credit_Score_Range,Employment_Status,Marital_Status,Dependents,Health_Condition,Fitness_Activity,Shopping_Habits,Travel_Habits,Loan_Status
0,18-25,Income_Low,Credit_Good,Employ_Self-Employed,Marital_Married,Dependents_None,Health_Acute,Fitness_Low,Shopping_Luxury,Travel_Frequently,Loan_Denied
1,51+,Income_Low,Credit_Poor,Employ_Employed,Marital_Married,Dependents_None,Health_Acute,Fitness_High,Shopping_Frugal,Travel_Frequently,Loan_Denied
2,51+,Income_Low,Credit_Excellent,Employ_Unemployed,Marital_Single,Dependents_None,Health_Acute,Fitness_Moderate,Shopping_Balanced,Travel_Occasionally,Loan_Denied
3,26-35,Income_Medium,Credit_Poor,Employ_Employed,Marital_Widowed,Dependents_None,Health_Chronic,Fitness_Moderate,Shopping_Luxury,Travel_Occasionally,Loan_Denied
4,51+,Income_High,Credit_Poor,Employ_Retired,Marital_Single,Dependents_3+,Health_Chronic,Fitness_High,Shopping_Luxury,Travel_Occasionally,Loan_Denied
5,18-25,Income_Low,Credit_Average,Employ_Unemployed,Marital_Single,Dependents_None,Health_Healthy,Fitness_Moderate,Shopping_Balanced,Travel_Occasionally,Loan_Denied
6,26-35,Income_Medium,Credit_Good,Employ_Self-Employed,Marital_Divorced,Dependents_3+,Health_Acute,Fitness_High,Shopping_Frugal,Travel_Frequently,Loan_Denied
7,26-35,Income_Low,Credit_Excellent,Employ_Retired,Marital_Divorced,Dependents_3+,Health_Acute,Fitness_High,Shopping_Frugal,Travel_Frequently,Loan_Denied
8,18-25,Income_Low,Credit_Poor,Employ_Unemployed,Marital_Widowed,1-2,Health_Healthy,Fitness_Low,Shopping_Luxury,Travel_Frequently,Loan_Denied
9,26-35,Income_High,Credit_Excellent,Employ_Retired,Marital_Widowed,Dependents_None,Health_Chronic,Fitness_Low,Shopping_Frugal,Travel_Frequently,Loan_Denied


In [38]:

# Data Preprocessing
# Before Applying the FPGrowth Algorithm, we need to preprocess the data
# One-Hot Encoding, Remember get dummies?

from mlxtend.preprocessing import TransactionEncoder

# Consolidate each transaction into a single list of items, removing NaN values
transactions = df.apply(lambda row: row.dropna().tolist(), axis = 1).tolist()

# Initialize TransactionEncoder
encoder = TransactionEncoder()

# Fit and transform the transactions data
transaction_matrix = encoder.fit_transform(transactions)

# Convert to DataFrame
transaction_df = pd.DataFrame(transaction_matrix, columns = encoder.columns_)
transaction_df

,1-2,18-25,26-35,36-50,51+,Credit_Average,Credit_Excellent,Credit_Good,Credit_Poor,Dependents_3+,...,Marital_Divorced,Marital_Married,Marital_Single,Marital_Widowed,Shopping_Balanced,Shopping_Frugal,Shopping_Luxury,Travel_Frequently,Travel_Occasionally,Travel_Rarely
0,False,True,False,False,False,False,False,True,False,False,...,False,True,False,False,False,False,True,True,False,False
1,False,False,False,False,True,False,False,False,True,False,...,False,True,False,False,False,True,False,True,False,False
2,False,False,False,False,True,False,True,False,False,False,...,False,False,True,False,True,False,False,False,True,False
3,False,False,True,False,False,False,False,False,True,False,...,False,False,False,True,False,False,True,False,True,False
4,False,False,False,False,True,False,False,False,True,True,...,False,False,True,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,False,True,False,False,False,False,True,False,False,True,...,False,False,True,False,True,False,False,True,False,False
996,False,False,False,False,True,False,False,False,True,False,...,False,False,False,True,False,True,False,False,False,True
997,False,False,False,True,False,True,False,False,False,True,...,False,True,False,False,False,True,False,True,False,False
998,True,False,False,True,False,True,False,False,False,False,...,False,True,False,False,False,False,True,False,False,True


In [39]:

# Appying the FPGrowth Algorithm
# Since data are cleaned and prepared for frequent itemset

from mlxtend.frequent_patterns import fpgrowth, association_rules

# Apply the FPGrowth Algorithm
frequent_itemsets = fpgrowth(transaction_df, min_support = 0.1, use_colnames = True)

# min_support is the minimum support threshold. Itemsets with support greater than or equal to this threshold will be returned.
#use_colnames = True ensures that the item names are used in the output instead of column indices.


In [40]:

# View Frequent Itemsets

print(frequent_itemsets)

     support                                      itemsets
0      0.970                                 (Loan_Denied)
1      0.339                             (Dependents_None)
2      0.335                                 (Fitness_Low)
3      0.331                                (Health_Acute)
4      0.327                           (Travel_Frequently)
..       ...                                           ...
317    0.119   (Travel_Rarely, Dependents_3+, Loan_Denied)
318    0.113    (Travel_Rarely, Fitness_High, Loan_Denied)
319    0.122  (Travel_Rarely, Loan_Denied, Health_Healthy)
320    0.110     (Fitness_Low, Travel_Rarely, Loan_Denied)
321    0.216                          (36-50, Loan_Denied)

[322 rows x 2 columns]


In [41]:

# Generate Association Rules

rules = association_rules(frequent_itemsets, num_itemsets = len(transaction_df), metric = "confidence", min_threshold = 0.1)

rules.loc[:, :'lift']
# rules.loc[:, :'lift'].to_csv('rules.csv')

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,(Loan_Denied),(Dependents_None),0.970,0.339,0.317,0.326804,0.964024
1,(Dependents_None),(Loan_Denied),0.339,0.970,0.317,0.935103,0.964024
2,(Shopping_Frugal),(Dependents_None),0.345,0.339,0.114,0.330435,0.974734
3,(Dependents_None),(Shopping_Frugal),0.339,0.345,0.114,0.336283,0.974734
4,(Dependents_None),(Travel_Occasionally),0.339,0.342,0.127,0.374631,1.095413
...,...,...,...,...,...,...,...
1033,(Fitness_Low),"(Travel_Rarely, Loan_Denied)",0.335,0.331,0.110,0.328358,0.992019
1034,(Travel_Rarely),"(Fitness_Low, Loan_Denied)",0.331,0.335,0.110,0.332326,0.992019
1035,(Loan_Denied),"(Fitness_Low, Travel_Rarely)",0.970,0.110,0.110,0.113402,1.030928
1036,(36-50),(Loan_Denied),0.221,0.970,0.216,0.977376,1.007604


In [46]:
# Find the rule with the highest lift
most_pressing_rule = rules.loc[rules['lift'].idxmax()]


In [47]:
# Display the most pressing rule
print("The most pressing association rule is:")
print(most_pressing_rule)

The most pressing association rule is:
antecedents           (Loan_Denied, Dependents_None)
consequents                      (Employ_Unemployed)
antecedent support                             0.317
consequent support                             0.253
support                                        0.102
confidence                                  0.321767
lift                                        1.271805
representativity                                 1.0
leverage                                    0.021799
conviction                                  1.101391
zhangs_metric                               0.312907
jaccard                                     0.217949
certainty                                   0.092057
kulczynski                                  0.362464
Name: 706, dtype: object


In [43]:
# Find the rule with the highest lift for Loan_Denied
rules_denied = rules[rules['consequents'].apply(lambda x: 'Loan_Denied' in x)]
most_pressing_rule_denied = rules_denied.loc[rules_denied['lift'].idxmax()]

In [44]:
# Display the most pressing rule for loan disapproval
print("The most pressing association rule for loan disapproval is:")
print(most_pressing_rule_denied)

The most pressing association rule for loan disapproval is:
antecedents                      (Employ_Unemployed)
consequents           (Loan_Denied, Dependents_None)
antecedent support                             0.253
consequent support                             0.317
support                                        0.102
confidence                                  0.403162
lift                                        1.271805
representativity                                 1.0
leverage                                    0.021799
conviction                                  1.144364
zhangs_metric                               0.286099
jaccard                                     0.217949
certainty                                   0.126152
kulczynski                                  0.362464
Name: 707, dtype: object


In [50]:
approved_loans_df = df[df['Loan_Status'] == 'Loan_Approved']
display(approved_loans_df)

,Age_Group,Income_Level,Credit_Score_Range,Employment_Status,Marital_Status,Dependents,Health_Condition,Fitness_Activity,Shopping_Habits,Travel_Habits,Loan_Status
